# ICS 604: APPLIED DATA SCIENCE

## Bayesian Parameter Estimation
---

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

Recall that, in an A/B test, we compare two versions of a webpage to see which performs better at a specific task $X$. For example, we might want to determine whether Version A (new) or Version B (old) is more effective at getting visitors to sign up for a newsletter.

<center><img src="https://www.dropbox.com/scl/fi/rnvqswb1yjkkvvlvsouk0/ab.jpg?rlkey=twogus8zpr53zpq9u59fgto5r&st=6qq3jngj&dl=1" alt="drawing" style="width:600px"></center>

## Bayesian Parameter Estimation

In Bayesian parameter estimation, we start by considering a generative model that describes how data are produced. For the A/B testing experiment from the previous slide, this involves modeling the behavior of a user interacting with a webpage prototype. For a single person and one version of the webpage, the generative process can be illustrated by assuming the user is equally likely to sign up or forgo signing up, with probability $p=0.5$. Each outcome — signing up or not — can be seen as a random draw from this underlying probability distribution.

<center><img src="https://www.dropbox.com/scl/fi/jjzbcrutgjj1ie8alg396/ab_user.jpg?rlkey=dfy28z1mm08kgh1zzvz4ux4oa&st=psp02gl3&dl=1" width="400"></center>

<br>
Extending this idea to multiple users, such as selecting 8 individuals to try Version A, the generative process conceptually repeats for each person. Each user’s outcome is treated as an independent draw from the same probabilistic model, reflecting the same underlying likelihood of signing up. This framework provides the basis for thinking about how the observed data are generated before any parameter estimation is performed.

### The Long Run Distribution

When the experiment is repeated many times, the outcomes form a binomial distribution, representing the number of successes observed in each set of independent trials. For instance, with 8 individuals and a signup probability of $p=0.5$, the distribution is fully described by these parameters, $n=8$ and $p=0.5$. The graph below illustrates how the number of sign-ups varies across repeated experiments, highlighting the variability inherent in small samples.

<center><img src="https://www.dropbox.com/scl/fi/dp99r0f3evkvjeoh5kde1/longrun.png?rlkey=dubu6ccj00hjq16a84nedd9gm&st=1p6ouw5d&dl=1" width="650"></center>

### Observations

In reality, the true signup rates $p_A$ and $p_B$ for Version A and B are unknown. A common approach is to estimate these parameters using maximum likelihood (ML). However, ML has limitations: with a small sample, random fluctuations can heavily skew the estimates, and confidence intervals only indicate where the true parameter might lie with 95% confidence rather than providing probabilities for individual values.

Furthermore, ML does not allow us to incorporate prior knowledge about the parameter. For example, if previous research suggests a parameter should be around 0.9, there is no straightforward way to include that belief in the estimation process. Any practical approach should account for both the observed data and prior knowledge, which motivates a Bayesian perspective for parameter estimation.

In [ ]:
# Suppose outcomes are 
outcomes = [True] * 5 + [False] * 3
outcomes

In [ ]:
bootstrap = np.random.choice(outcomes, size=8, replace=True)
bootstrap_proportion = sum(bootstrap)/len(bootstrap)
bootstrap_proportion

In [ ]:
bootstrap_proportions = []
for _ in range(10_000):
    bootstrap = np.random.choice(outcomes, size=8, replace=True)
    bootstrap_proportion = sum(bootstrap)/len(bootstrap)
    bootstrap_proportions.append(bootstrap_proportion)
np.percentile(bootstrap_proportions, (2.5, 97.5))

In [ ]:
bootstrap = np.random.choice([True, False], size=8, replace=True)
bootstrap_proportion = sum(bootstrap)/len(bootstrap)
print(bootstrap)
bootstrap_proportion

In [ ]:
bootstrap_proportions = []
for _ in range(10000):
    bootstrap = np.random.choice([True, False], size=8, replace=True)
    bootstrap_proportion = sum(bootstrap)/len(bootstrap)
    bootstrap_proportions.append(bootstrap_proportion)
np.percentile(bootstrap_proportions, (2.5, 97.5))

### Running the Simulation Backwards

Rather than generating random outcomes from a fixed parameter, we can approach the problem **in reverse**: we ask which parameter values could have plausibly produced the observed data. For example, if we observed 5 signups out of 8 trials, we want to know which values of $p_A$, the probability of signup for Version A, could have generated this outcome. Since $p_A$ is continuous between 0 and 1, there is a range of possible values to consider.

### Bayesian Inference Using Rejection Sampling 

To estimate the most likely value of $p_A$, we can use **rejection sampling**, a simple Bayesian inference method. The idea is to simulate candidate parameter values and retain only those that are consistent with the observed results. This procedure approximates the **posterior distribution** of $p_A$, reflecting our updated beliefs after seeing the data.

### Rejection Sampling 

1. **Sample a parameter value** from the set of possible $p_A$ values. This initial distribution, known as the **prior**, encodes our beliefs about $p_A$ before seeing the data. In this example, we assume a uniform prior over the interval [0,1], meaning any value is equally likely.

2. **Generate an outcome** using the sampled parameter. Here, we simulate the experiment according to a Binomial distribution with the chosen $p_A$ and 8 trials.

3. **Compare the generated outcome to the observed data.** If the simulated result matches the actual observation (e.g., 5 signups), we record the parameter $p_A$ as a possible value. If it does not match, we reject it.

4. **Repeat the process many times.** Performing this procedure over many iterations produces a distribution of accepted $p_A$ values. This distribution approximates the posterior, showing which parameter values are most consistent with the observed data.

### Approximate Bayes Using Rejection Sampling

<img src="https://www.dropbox.com/scl/fi/ljlgdhkso7l6b52jzrcev/rejection.jpg?rlkey=0bfgx2szj9g18go02fm8ws2dz&st=2aw6jnb4&dl=1" width="1200">


### Exploring the Distribution of the Parameter $p$

After applying the rejection sampling procedure, we can examine the distribution of the accepted values of $p_A$. This distribution represents the parameter values that are most consistent with the observed data. In other words, it reflects what we believe the value of $p_A$ could plausibly be after observing the results of the experiment. Rather than identifying a single best value, this approach provides a range of possible values, with some appearing more frequently because they are more likely to generate the observed outcome.

Initially, our **prior belief** assumed that $p_A$ could take any value between 0 and 1 with equal probability. However, once the observed data are taken into account, some values of $p_A$ appear more often than others in the accepted samples. This indicates that these values are more consistent with the data. In principle, we could also specify a prior distribution that reflects stronger beliefs about certain values — for example, assigning higher probability to values we believe are more plausible based on previous knowledge or research.

The distribution that results after combining the prior belief with the observed data is called the **posterior distribution** of the parameter $p_A$. This posterior distribution summarizes our updated belief about the parameter after observing the data.

In [ ]:
# import packages
import numpy as np
import matplotlib.pyplot as plt

# Version A
n_A = 8        # number of people Version A was tested on
obs_A = 5      # observed number of signups for Version A
p_A_recorded = []

num_of_recorded_vals = 100_000

for _ in range(num_of_recorded_vals):
  # Sample a parameter value p_A
  p_A = np.random.uniform(0, 1)

  # For the p_A, generate an outcome from the Binomal distribution 
  v_A = np.random.binomial(n_A, p_A)
    
  # Compare generated outcomes to the observed data
  if v_A == obs_A:
    p_A_recorded.append(p_A)

In [ ]:
len(p_A_recorded)

In [ ]:
plt.style.use(["seaborn-v0_8-darkgrid"])

plt.figure(figsize=(8, 4)) 
plt.hist(p_A_recorded, bins=20, alpha=0.75, color="#A60628", edgecolor='black', linewidth=1.2)
plt.xlim(0, 1)
plt.xlabel("Signup rate ($p_A$)")
plt.title("Number of Recorded Values for Version A")

### Analyzing the Simulation Results

The histogram of the accepted values of $p_A$ shows that the probability is highest around approximately 
0.625. This result aligns with what we would expect from the likelihood, since the observed data suggest a signup rate near this value. At the same time, the distribution also indicates that other values of $p_A$ remain plausible, as they appear with non-negligible probability in the simulation results. Rather than pointing to a single estimate, the distribution reflects a range of parameter values that could reasonably explain the observed data.

In this particular example, the posterior distribution does not differ substantially from the likelihood because the prior assumption was uniform over the interval [0, 1], meaning we did not favor any parameter value before observing the data. As a result, the posterior largely mirrors the information contained in the observed outcomes.

Now imagine that we collect additional data for Version A. Suppose that in another experiment, 6 out of 9 users sign up for the same version. Incorporating this new dataset allows us to further update our beliefs about the parameter $p_A$, refining the distribution to reflect the combined evidence from both sets of observations.

### Updating Our Beliefs

So far, we have used the observed data to generate a posterior distribution for the parameter $p_A$. This posterior reflects our updated belief about the possible values of $p_A$ after observing the initial dataset.

When new data become available, we do not need to start the inference process from scratch. Instead, we can use the previously obtained posterior distribution as our new prior. In other words, rather than sampling $p_A$ uniformly from the interval [0, 1], we now sample parameter values from the posterior distribution derived from the earlier analysis. This reflects the idea that our prior knowledge has been updated by the data we have already observed.

In practice, this can be done by sampling directly from the histogram that represents the posterior distribution. Using these samples as the prior for the next round of inference allows us to incorporate additional data and progressively refine our estimate of $p_A$.

In [ ]:
np.random.choice(p_A_recorded)

In [ ]:
# import packages
import numpy as np
import matplotlib.pyplot as plt

# Version A
n_A = 9        # number of people Version A was tested on
obs_A = 6      # observed number of signups for Version A
p_A_recorded_experiment_2 = []

num_of_recorded_vals = 100_000

for _ in range(num_of_recorded_vals):
  # Sample a parameter value p_A from the posterior 
  p_A = np.random.choice(p_A_recorded)

  # For the p_A, generate an outcome from the Binomal distribution 
  v_A = np.random.binomial(n_A, p_A)
    
  # Compare generated outcomes to the observed data
  if v_A == obs_A:
    p_A_recorded_experiment_2.append(p_A)

In [ ]:
plt.figure(figsize=(8, 4)) 

plt.hist(p_A_recorded, bins=20, density=True, alpha=0.75, 
         label="Experiment 1", color="#A60628", edgecolor='black', linewidth=1.2,)
plt.hist(p_A_recorded_experiment_2, bins=20, density=True, alpha=0.75, 
         label="Experiment 2", color="#bcadd1", edgecolor='black', linewidth=1.2)
plt.xlim(0, 1)
plt.xlabel("Signup rate ($p_A$)")
plt.title("Frequency of Recorded Values for Experiment 1 and 2")
plt.legend(loc='upper left', fontsize=14)

In [ ]:
# import packages
import numpy as np
import matplotlib.pyplot as plt

# Version A
n_A = 8        # number of people Version A was tested on
obs_A = 6      # observed number of signups for Version A
p_A_recorded_experiment_3 = []

num_of_recorded_vals = 100_000

for _ in range(num_of_recorded_vals):
  # Sample a parameter value p_A from the posterior 
  p_A = np.random.choice(p_A_recorded_experiment_2)

  # For the p_A, generate an outcome from the Binomal distribution 
  v_A = np.random.binomial(n_A, p_A)
    
  # Compare generated outcomes to the observed data
  if v_A == obs_A:
    p_A_recorded_experiment_3.append(p_A)

In [ ]:
plt.figure(figsize=(8, 4)) 

plt.hist(p_A_recorded, bins=20, density=True, alpha=0.75, 
         label="Experiment 1", color="#a60628", edgecolor='black', linewidth=1.2)
plt.hist(p_A_recorded_experiment_2, bins=20, density=True, alpha=0.75, 
         label="Experiment 2", color="#bcadd1", edgecolor='black', linewidth=1.2)
plt.hist(p_A_recorded_experiment_3, bins=20, density=True, alpha=0.4, 
         label="Experiment 3", color="#1f5ae6", edgecolor='black', linewidth=1.2)
plt.xlim(0, 1)
plt.xlabel("Signup rate ($p_A$)")
plt.title("Frequency of Recorded Values for Three Experiments")
plt.legend(loc='upper left', fontsize=14);

In [ ]:
from scipy.stats import gaussian_kde

kde1 = gaussian_kde(p_A_recorded, bw_method=0.8)
kde2 = gaussian_kde(p_A_recorded_experiment_2, bw_method=0.8)
kde3 = gaussian_kde(p_A_recorded_experiment_3, bw_method=0.8)

In [ ]:
x = np.linspace(0, 1, 1000)

# Plot KDE for each experiment
fig = plt.figure(figsize=(8, 4))

plt.plot(x, kde1(x), label="Experiment 1", color="#a60628", linewidth=2)
plt.plot(x, kde2(x), label="Experiment 2", color="#bcadd1", linewidth=2)
plt.plot(x, kde3(x), label="Experiment 3", color="#1f5ae6", linewidth=2, alpha=0.75)

plt.xlim(0, 1)
plt.xlabel("Signup rate ($p_A$)")
plt.title("Kernel Density Estimation for Three Experiments")
plt.legend(loc='upper left', fontsize=14)
plt.style.use(["seaborn-v0_8-darkgrid"])

### Bayesian Inference for Parameter Estimation

Bayesian inference provides a framework for estimating model parameters by treating them as random variables and assigning probabilities to their possible values. Instead of identifying a single fixed estimate, this approach allows us to describe our uncertainty about a parameter through a probability distribution. In doing so, it enables us to represent our beliefs about the parameter in a principled, probabilistic manner.

An important advantage of Bayesian inference is that it naturally incorporates prior knowledge. Prior beliefs about a parameter can be encoded in a prior distribution, which is then updated using observed data. As new data become available, the prior distribution is combined with the likelihood of the observed data to produce a posterior distribution. This process allows new evidence to be integrated systematically while leveraging insights and discoveries obtained from earlier observations.